# Clasificación de manzanas · YOLO + OpenCV sobre video

Este cuaderno hace cuatro cosas:
1. Descarga **Apple Sorting v2** de Roboflow y unifica `apple` y `damaged_apple` en una sola clase: `manzana`.
2. Hace el **fine-tuning** de un YOLO preentrenado.
3. Procesa `coral_manzanas.mp4` cuadro a cuadro con **OpenCV**, con **ByteTrack** para mantener el ID de cada manzana.
4. Mide si el pipeline alcanza **tiempo real** (FPS de procesamiento contra FPS del video).

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

Para la demo completa con las dos CNN, usá `notebooks/03_video_y_camara.ipynb`. El video de este notebook corresponde a la prueba inicial de detección.

In [2]:
!pip install -q ultralytics==8.4.192 roboflow

In [3]:
import torch
print("GPU disponible:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

GPU disponible: True | Tesla T4


## 1. Dataset

La clave de Roboflow va en los **Secrets** de Colab (ícono de llave, a la izquierda) con el nombre `ROBOFLOW_API_KEY`, así no queda escrita en el cuaderno.
La encontrás en Roboflow → Settings → API Keys.

In [4]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("arfiani-nur-sayidah-9lizr").project("apple-sorting-2bfhk")
dataset = project.version(2).download("yolov8")   # formato compatible con Ultralytics
DATA_DIR = dataset.location
print(DATA_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Apple-Sorting-2 in yolov8:: 100%|██████████| 1406/1406 [00:00<00:00, 11245.45it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
/content/Apple-Sorting-2


### Unificar clases: `apple` + `damaged_apple` → `manzana`

El detector solo tiene que encontrar manzanas. Si están dañadas o no lo decide después la CNN 2.

In [10]:
import glob, os, yaml

cambiados = 0
for split in ["train", "valid", "test"]:
    for f in glob.glob(f"{DATA_DIR}/{split}/labels/*.txt"):
        with open(f) as fh:
            lineas = [l.split() for l in fh if l.strip()]
        with open(f, "w") as fh:
            for l in lineas:
                fh.write(" ".join(["0"] + l[1:]) + "\n")
        cambiados += 1

cfg = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["manzana"],
}
DATA_YAML = f"{DATA_DIR}/data_manzana.yaml"
with open(DATA_YAML, "w") as fh:
    yaml.safe_dump(cfg, fh, allow_unicode=True)

for split in ["train", "valid", "test"]:
    print(split, len(glob.glob(f"{DATA_DIR}/{split}/images/*")), "imágenes")
print("archivos de etiquetas reescritos:", cambiados)

train 489 imágenes
valid 178 imágenes
test 30 imágenes
archivos de etiquetas reescritos: 697


## 2. Fine-tuning

`yolo11n` es el modelo más chico y rápido; conviene para tiempo real. Si después sobra margen de FPS, prueben `yolo11s`.

In [11]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # preentrenado en COCO
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,          # corta antes si deja de mejorar en validación
    project="runs",
    name="manzanas",
    exist_ok=True,
)

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Apple-Sorting-2/data_manzana.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=manzanas, nbs=64, nms=None, opset=None, optimize

In [ ]:
from pathlib import Path

BEST = str(model.trainer.best)
assert Path(BEST).is_file(), 'El entrenamiento no guardó best.pt.'
print('Uso:', BEST)

In [14]:
# Evaluación en el split de prueba (mAP50, mAP50-95, precisión, recall)
model = YOLO(BEST)
metrics = model.val(data=DATA_YAML, split="test")
print(f"mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f} | P: {metrics.box.mp:.3f} | R: {metrics.box.mr:.3f}")

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 662.2±143.5 MB/s, size: 17.9 KB)
val: Scanning /content/Apple-Sorting-2/test/labels... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 1.9Kit/s 0.0s
val: New cache created: /content/Apple-Sorting-2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.4s
                   all         30        157       0.91      0.892       0.96       0.66
Speed: 4.8ms preprocess, 11.7ms inference, 0.0ms loss, 6.7ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.960 | mAP50-95: 0.660 | P: 0.910 | R: 0.892


Métricas guardadas de la corrida del grupo en Apple Sorting v2. El cambio posterior para seleccionar `best.pt` se aplica al volver a entrenar.

In [ ]:
# Opcional: guardar los pesos en Drive para no reentrenar cada vez
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/manzanas
!cp {BEST} /content/drive/MyDrive/manzanas/yolo_manzanas_best.pt

## 3. Video con OpenCV + ByteTrack

Subí `coral_manzanas.mp4` con el panel de archivos (carpeta, a la izquierda) o dejalo en Drive y cambiá la ruta.

In [7]:
VIDEO = "/content/coral_manzanas.mp4"            # o "/content/drive/MyDrive/manzanas/coral_manzanas.mp4"
SALIDA = "/content/coral_manzanas_salida.mp4"

In [15]:
import cv2, time
import numpy as np

model = YOLO(BEST)

def recortar(frame_bgr, cajas, size=224):
    """Recortes listos para las CNN: 224x224 y en RGB (OpenCV entrega BGR)."""
    out = []
    h, w = frame_bgr.shape[:2]
    for x1, y1, x2, y2 in cajas:
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            out.append(None); continue
        rec = cv2.resize(frame_bgr[y1:y2, x1:x2], (size, size))
        out.append(cv2.cvtColor(rec, cv2.COLOR_BGR2RGB))
    return out

def clasificar(recortes):
    """Lugar para CNN 1 (variedad) y CNN 2 (estado). Por ahora no clasifica."""
    return [("VARIEDAD", "ESTADO") for _ in recortes]

cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f"No se pudo abrir {VIDEO}"
fps_video = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer = cv2.VideoWriter(SALIDA, cv2.VideoWriter_fourcc(*"mp4v"), fps_video, (W, H))
print(f"{W}x{H} a {fps_video:.1f} fps, {total} cuadros")

ids_vistos = set()
t_proc, n = 0.0, 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()

    r = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.4, verbose=False)[0]
    cajas, ids = [], []
    if r.boxes.id is not None:
        cajas = r.boxes.xyxy.cpu().numpy().astype(int).tolist()
        ids = r.boxes.id.cpu().numpy().astype(int).tolist()
    etiquetas = clasificar(recortar(frame, cajas))

    t_proc += time.perf_counter() - t0   # solo procesamiento, sin dibujo ni escritura
    n += 1

    for (x1, y1, x2, y2), tid, (var, est) in zip(cajas, ids, etiquetas):
        ids_vistos.add(tid)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (193, 211, 143), 3)
        texto = f"ID {tid} | {var} | {est}"
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 8, y1), (193, 211, 143), -1)
        cv2.putText(frame, texto, (x1 + 4, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (107, 48, 18), 2)
    writer.write(frame)

cap.release(); writer.release()

fps_proc = n / t_proc if t_proc else 0
print(f"Cuadros: {n} | IDs distintos: {len(ids_vistos)}")
print(f"Video: {fps_video:.1f} fps | Pipeline: {fps_proc:.1f} fps | Latencia media: {1000 * t_proc / max(n, 1):.1f} ms/cuadro")
print("¿Tiempo real?", "SÍ" if fps_proc >= fps_video else "NO")

960x540 a 60.0 fps, 6721 cuadros
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 149ms
Prepared 1 package in 22ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Cuadros: 6720 | IDs distintos: 23
Video: 60.0 fps | Pipeline: 68.9 fps | Latencia media: 14.5 ms/cuadro
¿Tiempo real? SÍ


## 4. Ver el resultado

`mp4v` no se reproduce en el navegador, así que se convierte a H.264 para verlo acá. Para descargarlo, usá el panel de archivos.

In [ ]:
!ffmpeg -y -loglevel error -i {SALIDA} -vcodec libx264 -pix_fmt yuv420p /content/coral_manzanas_web.mp4
from IPython.display import Video
Video("/content/coral_manzanas_web.mp4", embed=True, width=800)